# Model Development and Saving

This notebook develops the final Hybrid Random Forest QSAR model for permeability prediction.

Unlike previous notebooks focused on model evaluation and validation, this notebook uses the complete dataset to train a production-ready model that can later be deployed as a prediction application.

The workflow includes:

- Hybrid feature generation
- Final model training
- Model saving
- Applicability Domain asset preparation
- Deployment-ready prediction pipeline preparation

## Import Required Libraries

The required cheminformatics, machine learning, and model persistence libraries are imported.

These libraries will be used to generate molecular features, train the final model, and save deployment-ready assets.

In [2]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors

from sklearn.ensemble import RandomForestRegressor

import joblib

## Load Dataset

The complete permeability dataset is loaded.

All available molecules will be used for final model development to maximize the information available to the model.

In [6]:
df = pd.read_csv(
    "../Data_set/final_12k_log_transformed_papp_dataset.csv"
)

print(df.shape)

df.head()

(12290, 3)


,canonical_smiles,standard_value,log_papp
0,Br.Cc1c2c(cc[n+]1Cc1ccccc1)c1ccc(OCC(=O)OCCCCO...,0.05,-1.301030
1,Brc1ccc(-c2nc3ccc(Br)cn3n2)cc1,3.49,0.542825
2,Brc1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)cc1,5.90,0.770852
3,Brc1ccc(C2(CC3CCCC3)c3ccccc3-c3nccn32)cn1,25.00,1.397940
4,Brc1ccc(C2(CC3CCOCC3)c3ccccc3-c3nccn32)cc1,31.00,1.491362


In [3]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

## Generating Hybrid Molecular Features

Hybrid molecular features are generated by combining:

- Morgan fingerprints (2048 bits)
- Physicochemical molecular descriptors

This representation captures both structural and physicochemical information relevant to permeability prediction.

In [7]:
def generate_hybrid_features(df):

    fingerprints = []
    descriptors_list = []

    for smi in df['canonical_smiles']:

        mol = Chem.MolFromSmiles(smi)

        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=2,
            nBits=2048
        )

        fingerprints.append(np.array(fp))

        descriptors_list.append([
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.NumRotatableBonds(mol)
        ])

    fp_array = np.array(fingerprints)

    desc_array = np.array(descriptors_list)

    X = np.hstack([fp_array, desc_array])

    return X

## Creating Final Feature Matrix

Hybrid molecular features are generated for the complete dataset.

These features will be used to train the final production-ready model.

In [8]:
X = generate_hybrid_features(df)

y = df['log_papp'].values

print(X.shape)

print(y.shape)

(12290, 2054)
(12290,)


## Training the Final Hybrid Random Forest Model

The final Hybrid Random Forest model is trained using the complete dataset.

Unlike previous validation experiments, all available molecules are used to maximize learning and create a deployment-ready model.

The trained model will later be saved and integrated into a prediction pipeline.

In [9]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X, y)

print("Final Hybrid RF model training completed.")

Final Hybrid RF model training completed.


## Evaluating Training Performance

The model is evaluated on the complete training dataset to verify that training was successful.

These metrics are not intended to estimate generalization performance because validation studies have already been performed in previous notebooks.

In [10]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

y_pred = rf_model.predict(X)

rmse = np.sqrt(
    mean_squared_error(y, y_pred)
)

r2 = r2_score(
    y,
    y_pred
)

print("Training RMSE:", rmse)

print("Training R²:", r2)

Training RMSE: 0.20165513145099673
Training R²: 0.945054477969776


## Saving the Final Hybrid Random Forest Model

The final Hybrid Random Forest model is saved as a serialized file using Joblib.

The saved model can later be loaded without retraining and integrated into prediction pipelines, web applications, and deployment environments.

In [11]:
import joblib

joblib.dump(
    rf_model,
    "hybrid_rf_model.pkl"
)

print("Model saved successfully.")

Model saved successfully.


## Verifying Saved Model

The saved model is reloaded to confirm that serialization was successful and that the model can be reused without retraining.

In [12]:
loaded_model = joblib.load(
    "hybrid_rf_model.pkl"
)

print(type(loaded_model))

<class 'sklearn.ensemble._forest.RandomForestRegressor'>


## Saving Applicability Domain Reference Data

The complete hybrid feature matrix is saved as a reference chemical space for future Applicability Domain calculations.

This reference dataset will allow new molecules to be compared against the training chemical space during deployment.

In [13]:
joblib.dump(
    X,
    "training_reference_features.pkl"
)

print("Training reference features saved.")

Training reference features saved.


In [14]:
metadata = {

    "model_name": "Hybrid Random Forest",

    "fingerprint_bits": 2048,

    "descriptors": 6,

    "total_features": 2054,

    "training_samples": 12290
}

joblib.dump(
    metadata,
    "model_metadata.pkl"
)

print("Metadata saved.")

Metadata saved.


# Model Development Summary

The final Hybrid Random Forest model was successfully trained using the complete permeability dataset.

Deployment assets generated:

- hybrid_rf_model.pkl
- training_reference_features.pkl
- model_metadata.pkl

These assets will be used in the next phase to construct a prediction pipeline and deploy a permeability prediction application.